# Multi-Task SetFit for VeriPromiseESG

This notebook is a standalone SetFit-style training and inference flow. It does not modify the original `model_train.ipynb`, `model_inference.ipynb`, `compare_result.py`, or data files.

Design choices:
- Keep the original fold CSV input format and final submission CSV output format.
- Keep the original hierarchical routing: T1 controls T2-T4, and T3 controls T4.
- Use one shared SentenceTransformer encoder for T1-T4 contrastive fine-tuning, then train four lightweight task heads on the shared embedding space.
- Use a Chinese sentence embedding model by default: `BAAI/bge-large-zh-v1.5`, with `BAAI/bge-base-zh-v1.5` as fallback.

References:
- SetFit docs: https://huggingface.co/docs/setfit/index
- BGE zh model: https://huggingface.co/BAAI/bge-large-zh-v1.5
- SetFit paper: https://arxiv.org/abs/2209.11055


In [ ]:
# Install dependencies. Restart the runtime/kernel after this cell if your environment asks for it.
!pip install -q setfit sentence-transformers datasets scikit-learn pandas numpy tqdm joblib


In [ ]:
import gc
import json
import os
import random
import warnings
from dataclasses import dataclass
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import torch
from sentence_transformers import InputExample, SentenceTransformer, losses
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, f1_score
from sklearn.preprocessing import LabelEncoder
from torch.utils.data import DataLoader
from tqdm.auto import tqdm

try:
    import setfit
except Exception:
    setfit = None

warnings.filterwarnings("ignore")


In [ ]:
# ==========================================
# 0. Configuration
# ==========================================

RAW_BASE_URL = "https://raw.githubusercontent.com/Heng1222/VeriPromiseESG_2026_TEAM_9906/main/app/data/clean_data/"
USE_GITHUB_RAW = True

# Default mirrors the original inference notebook, which uses val_fold_1 as a test target.
# Change this to the actual test CSV when producing a challenge submission.
TEST_CSV_PATH = f"{RAW_BASE_URL}val_fold_1.csv"

FOLDS = [1, 2, 3, 4, 5]
SEED = 42

PRIMARY_MODEL_NAME = "BAAI/bge-large-zh-v1.5"
FALLBACK_MODEL_NAME = "BAAI/bge-base-zh-v1.5"
FORCE_FALLBACK_MODEL = False

MAX_SEQ_LENGTH = 512
HEAD_RATIO = 0.25
PAIR_POS_PER_LABEL = 180
PAIR_NEG_PER_LABEL = 180
CONTRASTIVE_EPOCHS = 2
CONTRASTIVE_BATCH_SIZE = 16
HEAD_MAX_ITER = 3000
HEAD_C = 1.0
SYNTHETIC_ID_MIN = 90000
SYNTHETIC_T4_SAMPLE_WEIGHT = 0.35

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
OUTPUT_DIR = Path("setfit_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

OOF_PREDICTION_CSV = OUTPUT_DIR / "setfit_oof_predictions.csv"
OOF_PROBABILITY_CSV = OUTPUT_DIR / "setfit_oof_probabilities.csv"
THRESHOLD_JSON = OUTPUT_DIR / "setfit_thresholds.json"
SUBMISSION_CSV = Path("setfit_final_submission.csv")

print(f"Device: {DEVICE}")
print(f"SetFit package: {getattr(setfit, '__version__', 'not imported')}")


In [ ]:
# ==========================================
# 1. Task definitions and helpers
# ==========================================

ID_COLUMN = "id"
TEXT_COLUMN = "data"
ESG_COLUMN = "esg_type"

TARGET_COLUMNS = [
    "promise_status",
    "verification_timeline",
    "evidence_status",
    "evidence_quality",
]

MISSING_ALLOWED_COLUMNS = {
    "verification_timeline",
    "evidence_status",
    "evidence_quality",
}

TASK_PREFIX = {
    "t1": "任務：判斷是否有 ESG 承諾。",
    "t2": "任務：判斷承諾驗證時間。",
    "t3": "任務：判斷是否提供證據。",
    "t4": "任務：判斷證據品質。",
}

TASK_COLUMN = {
    "t1": "promise_status",
    "t2": "verification_timeline",
    "t3": "evidence_status",
    "t4": "evidence_quality",
}

TASK_CLASSES = {
    "t1": ["No", "Yes"],
    "t2": ["already", "within_2_years", "between_2_and_5_years", "longer_than_5_years"],
    "t3": ["No", "Yes"],
    "t4": ["Clear", "Not Clear", "Misleading"],
}


def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def find_project_root():
    cwd = Path.cwd().resolve()
    candidates = [cwd, *cwd.parents]
    for candidate in candidates:
        if (candidate / "app" / "data" / "clean_data").exists():
            return candidate
    return cwd


PROJECT_ROOT = find_project_root()
LOCAL_CLEAN_DATA_DIR = PROJECT_ROOT / "app" / "data" / "clean_data"
LOCAL_FINAL_SUBMISSION = PROJECT_ROOT / "final_submission.csv"


def read_fold_csv(fold_num, split):
    filename = f"{split}_fold_{fold_num}.csv"
    if USE_GITHUB_RAW:
        try:
            return pd.read_csv(f"{RAW_BASE_URL}{filename}")
        except Exception as exc:
            print(f"GitHub raw read failed for {filename}: {exc}. Falling back to local clean_data.")
    return pd.read_csv(LOCAL_CLEAN_DATA_DIR / filename)


def normalize_value(value):
    if pd.isna(value):
        return None
    value = str(value).strip()
    if value == "" or value.upper() == "N/A":
        return None
    if value == "more_than_5_years":
        return "longer_than_5_years"
    return value


def normalize_series(series):
    return series.apply(normalize_value)


def is_synthetic_row(row):
    try:
        return int(row[ID_COLUMN]) >= SYNTHETIC_ID_MIN
    except Exception:
        return False


def process_esg_type(esg_value):
    if pd.isna(esg_value) or str(esg_value).strip() == "":
        return "未知"
    parts = [part.strip() for part in str(esg_value).split(";") if part.strip()]
    return "、".join(parts) if parts else "未知"


def truncate_text_by_tokens(text, tokenizer, max_seq_length=MAX_SEQ_LENGTH, head_ratio=HEAD_RATIO):
    token_ids = tokenizer.encode(str(text), add_special_tokens=False)
    max_body_len = max_seq_length - 2
    if len(token_ids) <= max_body_len:
        return str(text)
    head_len = int(max_body_len * head_ratio)
    tail_len = max_body_len - head_len
    kept = token_ids[:head_len] + token_ids[-tail_len:]
    return tokenizer.decode(kept, skip_special_tokens=True, clean_up_tokenization_spaces=True)


def build_base_text(row, tokenizer=None):
    esg_text = process_esg_type(row.get(ESG_COLUMN, ""))
    raw_text = str(row.get(TEXT_COLUMN, ""))
    full_text = f"ESG類型：{esg_text}。文本：{raw_text}"
    if tokenizer is not None:
        full_text = truncate_text_by_tokens(full_text, tokenizer)
    return full_text


def build_task_text(task_key, row, tokenizer=None):
    return TASK_PREFIX[task_key] + build_base_text(row, tokenizer=tokenizer)


def validate_required_columns(df, name):
    required = [ID_COLUMN, TEXT_COLUMN, ESG_COLUMN] + TARGET_COLUMNS
    missing = [col for col in required if col not in df.columns]
    if missing:
        raise ValueError(f"{name} missing columns: {missing}")


In [ ]:
# ==========================================
# 2. Multi-task SetFit data construction
# ==========================================

def filter_task_dataframe(df, task_key, include_synthetic_for_t4=True):
    work = df.copy()
    for col in TARGET_COLUMNS:
        if col in work.columns:
            work[col] = normalize_series(work[col])

    if task_key != "t4" or not include_synthetic_for_t4:
        work = work[~work.apply(is_synthetic_row, axis=1)]

    if task_key == "t1":
        mask = work["promise_status"].notna()
    elif task_key == "t2":
        mask = (work["promise_status"] == "Yes") & work["verification_timeline"].notna()
    elif task_key == "t3":
        mask = (work["promise_status"] == "Yes") & work["evidence_status"].notna()
    elif task_key == "t4":
        mask = (
            (work["promise_status"] == "Yes")
            & (work["evidence_status"] == "Yes")
            & work["evidence_quality"].notna()
        )
    else:
        raise ValueError(f"Unknown task: {task_key}")

    filtered = work.loc[mask].copy()
    filtered = filtered[filtered[TASK_COLUMN[task_key]].isin(TASK_CLASSES[task_key])]
    return filtered.reset_index(drop=True)


def build_multitask_views(train_df, tokenizer):
    rows = []
    for task_key in ["t1", "t2", "t3", "t4"]:
        task_df = filter_task_dataframe(train_df, task_key, include_synthetic_for_t4=True)
        label_col = TASK_COLUMN[task_key]
        for _, row in task_df.iterrows():
            label = normalize_value(row[label_col])
            rows.append(
                {
                    "id": row[ID_COLUMN],
                    "task": task_key,
                    "text": build_task_text(task_key, row, tokenizer=tokenizer),
                    "label": f"{task_key.upper()}::{label}",
                    "is_synthetic": is_synthetic_row(row),
                }
            )
    views = pd.DataFrame(rows)
    if views.empty:
        raise ValueError("No training views were created.")
    return views


def sample_pairs_for_task(task_views, rng):
    examples = []
    labels = sorted(task_views["label"].unique())
    groups = {label: task_views.loc[task_views["label"] == label, "text"].tolist() for label in labels}

    for label in labels:
        texts = groups[label]
        if len(texts) >= 2:
            for _ in range(PAIR_POS_PER_LABEL):
                a, b = rng.choice(texts, size=2, replace=False)
                examples.append(InputExample(texts=[a, b], label=1.0))

        other_labels = [other for other in labels if other != label and groups[other]]
        if not other_labels or not texts:
            continue
        for _ in range(PAIR_NEG_PER_LABEL):
            other_label = rng.choice(other_labels)
            a = rng.choice(texts)
            b = rng.choice(groups[other_label])
            examples.append(InputExample(texts=[a, b], label=0.0))
    return examples


def build_contrastive_examples(multitask_views, seed=SEED):
    rng = np.random.default_rng(seed)
    all_examples = []
    for task_key in ["t1", "t2", "t3", "t4"]:
        task_views = multitask_views[multitask_views["task"] == task_key]
        all_examples.extend(sample_pairs_for_task(task_views, rng))
    rng.shuffle(all_examples)
    return all_examples


In [ ]:
# ==========================================
# 3. Model training: shared SetFit encoder + 4 heads
# ==========================================

def load_encoder():
    model_name = FALLBACK_MODEL_NAME if FORCE_FALLBACK_MODEL else PRIMARY_MODEL_NAME
    try:
        encoder = SentenceTransformer(model_name, device=DEVICE)
    except RuntimeError as exc:
        if model_name == PRIMARY_MODEL_NAME:
            print(f"Primary model failed to load: {exc}")
            print(f"Falling back to {FALLBACK_MODEL_NAME}")
            encoder = SentenceTransformer(FALLBACK_MODEL_NAME, device=DEVICE)
        else:
            raise
    encoder.max_seq_length = MAX_SEQ_LENGTH
    return encoder


def fine_tune_shared_encoder(encoder, train_df, fold, seed=SEED):
    multitask_views = build_multitask_views(train_df, tokenizer=encoder.tokenizer)
    examples = build_contrastive_examples(multitask_views, seed=seed + fold)
    print(f"Fold {fold}: multitask views={len(multitask_views)}, contrastive pairs={len(examples)}")
    print(multitask_views.groupby(["task", "label"]).size())

    train_loader = DataLoader(examples, shuffle=True, batch_size=CONTRASTIVE_BATCH_SIZE)
    train_loss = losses.CosineSimilarityLoss(encoder)
    warmup_steps = max(1, int(len(train_loader) * CONTRASTIVE_EPOCHS * 0.1))
    encoder.fit(
        train_objectives=[(train_loader, train_loss)],
        epochs=CONTRASTIVE_EPOCHS,
        warmup_steps=warmup_steps,
        show_progress_bar=True,
    )
    return encoder


def encode_task_texts(encoder, df, task_key, batch_size=32):
    texts = [build_task_text(task_key, row, tokenizer=encoder.tokenizer) for _, row in df.iterrows()]
    return encoder.encode(
        texts,
        batch_size=batch_size,
        normalize_embeddings=True,
        convert_to_numpy=True,
        show_progress_bar=False,
    )


def fit_task_head(encoder, train_df, task_key):
    task_df = filter_task_dataframe(train_df, task_key, include_synthetic_for_t4=True)
    labels = task_df[TASK_COLUMN[task_key]].apply(normalize_value).tolist()
    embeddings = encode_task_texts(encoder, task_df, task_key)

    label_encoder = LabelEncoder()
    y = label_encoder.fit_transform(labels)

    sample_weight = np.ones(len(task_df), dtype=float)
    if task_key == "t4":
        synthetic_mask = task_df.apply(is_synthetic_row, axis=1).to_numpy()
        sample_weight[synthetic_mask] = SYNTHETIC_T4_SAMPLE_WEIGHT

    clf = LogisticRegression(
        C=HEAD_C,
        class_weight="balanced",
        max_iter=HEAD_MAX_ITER,
        n_jobs=-1,
        random_state=SEED,
    )
    clf.fit(embeddings, y, sample_weight=sample_weight)

    print(f"{task_key}: trained head on {len(task_df)} rows, classes={list(label_encoder.classes_)}")
    return {"classifier": clf, "label_encoder": label_encoder}


def train_fold(fold, seed=SEED):
    set_seed(seed + fold)
    train_df = read_fold_csv(fold, "train")
    val_df = read_fold_csv(fold, "val")
    validate_required_columns(train_df, f"train_fold_{fold}")
    validate_required_columns(val_df, f"val_fold_{fold}")

    encoder = load_encoder()
    encoder = fine_tune_shared_encoder(encoder, train_df, fold=fold, seed=seed)

    heads = {task_key: fit_task_head(encoder, train_df, task_key) for task_key in ["t1", "t2", "t3", "t4"]}

    fold_dir = OUTPUT_DIR / f"fold_{fold}"
    fold_dir.mkdir(parents=True, exist_ok=True)
    encoder.save(str(fold_dir / "encoder"))
    joblib.dump(heads, fold_dir / "task_heads.joblib")

    del encoder
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return fold_dir, val_df


def load_fold_artifacts(fold):
    fold_dir = OUTPUT_DIR / f"fold_{fold}"
    encoder = SentenceTransformer(str(fold_dir / "encoder"), device=DEVICE)
    encoder.max_seq_length = MAX_SEQ_LENGTH
    heads = joblib.load(fold_dir / "task_heads.joblib")
    return encoder, heads


In [ ]:
# ==========================================
# 4. Prediction, routing, and metrics
# ==========================================

def predict_task_probabilities(encoder, heads, df, task_key):
    embeddings = encode_task_texts(encoder, df, task_key)
    head = heads[task_key]
    clf = head["classifier"]
    label_encoder = head["label_encoder"]
    raw_probs = clf.predict_proba(embeddings)

    class_probs = pd.DataFrame(0.0, index=df.index, columns=TASK_CLASSES[task_key])
    for class_index, label in enumerate(label_encoder.classes_):
        if label in class_probs.columns:
            encoded_index = int(np.where(label_encoder.classes_ == label)[0][0])
            class_probs[label] = raw_probs[:, encoded_index]
    return class_probs


def predict_all_probabilities_for_fold(fold, df):
    encoder, heads = load_fold_artifacts(fold)
    out = pd.DataFrame({ID_COLUMN: df[ID_COLUMN].values})
    for task_key in ["t1", "t2", "t3", "t4"]:
        probs = predict_task_probabilities(encoder, heads, df.reset_index(drop=True), task_key)
        for cls in TASK_CLASSES[task_key]:
            out[f"{task_key}__{cls}"] = probs[cls].values

    del encoder, heads
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return out


def average_probability_frames(probability_frames):
    if not probability_frames:
        raise ValueError("No probability frames to average.")
    base = probability_frames[0][[ID_COLUMN]].copy()
    prob_cols = [col for col in probability_frames[0].columns if col != ID_COLUMN]
    for col in prob_cols:
        base[col] = np.mean([frame[col].to_numpy() for frame in probability_frames], axis=0)
    return base


def argmax_from_probs(row, task_key):
    classes = TASK_CLASSES[task_key]
    values = np.array([row[f"{task_key}__{cls}"] for cls in classes], dtype=float)
    return classes[int(values.argmax())]


def route_predictions(prob_df, t1_threshold=0.5, t3_threshold=0.5):
    results = []
    for _, row in prob_df.iterrows():
        t1_yes_prob = float(row["t1__Yes"])
        t3_yes_prob = float(row["t3__Yes"])
        t1_pred = "Yes" if t1_yes_prob >= t1_threshold else "No"

        if t1_pred == "No":
            results.append(
                {
                    ID_COLUMN: row[ID_COLUMN],
                    "promise_status": "No",
                    "verification_timeline": "N/A",
                    "evidence_status": "N/A",
                    "evidence_quality": "N/A",
                }
            )
            continue

        t2_pred = argmax_from_probs(row, "t2")
        t3_pred = "Yes" if t3_yes_prob >= t3_threshold else "No"

        if t3_pred == "No":
            results.append(
                {
                    ID_COLUMN: row[ID_COLUMN],
                    "promise_status": "Yes",
                    "verification_timeline": t2_pred,
                    "evidence_status": "No",
                    "evidence_quality": "N/A",
                }
            )
            continue

        t4_pred = argmax_from_probs(row, "t4")
        results.append(
            {
                ID_COLUMN: row[ID_COLUMN],
                "promise_status": "Yes",
                "verification_timeline": t2_pred,
                "evidence_status": "Yes",
                "evidence_quality": t4_pred,
            }
        )
    return pd.DataFrame(results)[[ID_COLUMN] + TARGET_COLUMNS]


def strict_task_f1(true_df, pred_df, column, average="macro"):
    merged = true_df[[ID_COLUMN, column]].merge(pred_df[[ID_COLUMN, column]], on=ID_COLUMN, suffixes=("_true", "_pred"))
    y_true = normalize_series(merged[f"{column}_true"])
    y_pred = normalize_series(merged[f"{column}_pred"])
    mask = y_true.notna()
    labels = TASK_CLASSES[{v: k for k, v in TASK_COLUMN.items()}[column]]
    return f1_score(y_true[mask], y_pred[mask].fillna("N/A"), labels=labels, average=average, zero_division=0)


def evaluate_submission(true_df, pred_df):
    rows = []
    for column in TARGET_COLUMNS:
        rows.append(
            {
                "task": column,
                "macro_f1": strict_task_f1(true_df, pred_df, column, average="macro"),
                "weighted_f1": strict_task_f1(true_df, pred_df, column, average="weighted"),
            }
        )
    metrics = pd.DataFrame(rows)
    summary = pd.DataFrame(
        [
            {
                "task": "average",
                "macro_f1": metrics["macro_f1"].mean(),
                "weighted_f1": metrics["weighted_f1"].mean(),
            }
        ]
    )
    return pd.concat([metrics, summary], ignore_index=True)


def tune_thresholds(oof_true_df, oof_prob_df):
    best = {"score": -1.0, "t1_threshold": 0.5, "t3_threshold": 0.5}
    grid = np.round(np.arange(0.20, 0.801, 0.01), 2)
    for t1_threshold in grid:
        for t3_threshold in grid:
            pred_df = route_predictions(oof_prob_df, t1_threshold=t1_threshold, t3_threshold=t3_threshold)
            metrics = evaluate_submission(oof_true_df, pred_df)
            score = float(metrics.loc[metrics["task"] == "average", "macro_f1"].iloc[0])
            if score > best["score"]:
                best = {"score": score, "t1_threshold": float(t1_threshold), "t3_threshold": float(t3_threshold)}
    return best


def print_per_class_reports(true_df, pred_df):
    for column in TARGET_COLUMNS:
        task_key = {v: k for k, v in TASK_COLUMN.items()}[column]
        merged = true_df[[ID_COLUMN, column]].merge(pred_df[[ID_COLUMN, column]], on=ID_COLUMN, suffixes=("_true", "_pred"))
        y_true = normalize_series(merged[f"{column}_true"])
        y_pred = normalize_series(merged[f"{column}_pred"])
        mask = y_true.notna()
        labels = TASK_CLASSES[task_key]
        print(f"\n=== {column} ===")
        print(classification_report(y_true[mask], y_pred[mask].fillna("N/A"), labels=labels, zero_division=0))
        print(pd.DataFrame(confusion_matrix(y_true[mask], y_pred[mask].fillna("N/A"), labels=labels), index=labels, columns=labels))


In [ ]:
# ==========================================
# 5. Train all folds and create OOF probabilities
# ==========================================

oof_true_frames = []
oof_probability_frames = []

for fold in FOLDS:
    print(f"\n{'=' * 48}")
    print(f"Training Fold {fold}")
    print(f"{'=' * 48}")
    _, val_df = train_fold(fold, seed=SEED)
    val_df = val_df.reset_index(drop=True)
    fold_probs = predict_all_probabilities_for_fold(fold, val_df)
    oof_true_frames.append(val_df[[ID_COLUMN] + TARGET_COLUMNS].copy())
    oof_probability_frames.append(fold_probs)

oof_true_df = pd.concat(oof_true_frames, ignore_index=True)
oof_prob_df = pd.concat(oof_probability_frames, ignore_index=True)
oof_prob_df.to_csv(OOF_PROBABILITY_CSV, index=False)
print(f"Saved OOF probabilities to {OOF_PROBABILITY_CSV}")


In [ ]:
# ==========================================
# 6. Threshold tuning and OOF evaluation
# ==========================================

best_thresholds = tune_thresholds(oof_true_df, oof_prob_df)
print("Best thresholds:", best_thresholds)

with open(THRESHOLD_JSON, "w", encoding="utf-8") as f:
    json.dump(best_thresholds, f, ensure_ascii=False, indent=2)

oof_pred_df = route_predictions(
    oof_prob_df,
    t1_threshold=best_thresholds["t1_threshold"],
    t3_threshold=best_thresholds["t3_threshold"],
)
oof_pred_df.to_csv(OOF_PREDICTION_CSV, index=False)
print(f"Saved OOF predictions to {OOF_PREDICTION_CSV}")

oof_metrics = evaluate_submission(oof_true_df, oof_pred_df)
display(oof_metrics)
print_per_class_reports(oof_true_df, oof_pred_df)


In [ ]:
# ==========================================
# 7. Baseline comparison against the original final_submission.csv when available
# ==========================================

try:
    baseline_true_df = read_fold_csv(1, "val")[[ID_COLUMN] + TARGET_COLUMNS]
    if LOCAL_FINAL_SUBMISSION.exists():
        baseline_pred_df = pd.read_csv(LOCAL_FINAL_SUBMISSION)[[ID_COLUMN] + TARGET_COLUMNS]
        baseline_metrics = evaluate_submission(baseline_true_df, baseline_pred_df)
        baseline_metrics.insert(0, "model", "original_final_submission")

        setfit_fold1_prob = oof_prob_df[oof_prob_df[ID_COLUMN].isin(baseline_true_df[ID_COLUMN])].copy()
        setfit_fold1_pred = route_predictions(
            setfit_fold1_prob,
            t1_threshold=best_thresholds["t1_threshold"],
            t3_threshold=best_thresholds["t3_threshold"],
        )
        setfit_fold1_metrics = evaluate_submission(baseline_true_df, setfit_fold1_pred)
        setfit_fold1_metrics.insert(0, "model", "setfit_oof_fold_1")
        display(pd.concat([baseline_metrics, setfit_fold1_metrics], ignore_index=True))
    else:
        print(f"Baseline file not found: {LOCAL_FINAL_SUBMISSION}")
except Exception as exc:
    print(f"Baseline comparison skipped: {exc}")


In [ ]:
# ==========================================
# 8. Ensemble inference and export
# ==========================================

def ensemble_inference_and_export(test_csv_path=TEST_CSV_PATH, output_csv_path=SUBMISSION_CSV):
    test_df = pd.read_csv(test_csv_path)
    if ID_COLUMN not in test_df.columns or TEXT_COLUMN not in test_df.columns or ESG_COLUMN not in test_df.columns:
        raise ValueError(f"Test CSV must contain {ID_COLUMN}, {TEXT_COLUMN}, and {ESG_COLUMN}.")

    probability_frames = []
    for fold in FOLDS:
        print(f"Predicting with fold {fold}")
        probability_frames.append(predict_all_probabilities_for_fold(fold, test_df.reset_index(drop=True)))

    avg_probs = average_probability_frames(probability_frames)

    if THRESHOLD_JSON.exists():
        with open(THRESHOLD_JSON, "r", encoding="utf-8") as f:
            thresholds = json.load(f)
    else:
        thresholds = {"t1_threshold": 0.5, "t3_threshold": 0.5}

    final_output = route_predictions(
        avg_probs,
        t1_threshold=float(thresholds["t1_threshold"]),
        t3_threshold=float(thresholds["t3_threshold"]),
    )

    final_output[ID_COLUMN] = test_df[ID_COLUMN].values
    final_output = final_output[[ID_COLUMN] + TARGET_COLUMNS]
    final_output.to_csv(output_csv_path, index=False)
    print(f"Exported SetFit submission to {output_csv_path}")
    display(final_output.head())
    return final_output


setfit_submission = ensemble_inference_and_export(TEST_CSV_PATH, SUBMISSION_CSV)


In [ ]:
# ==========================================
# 9. Output format checks
# ==========================================

def validate_output_csv(output_csv_path=SUBMISSION_CSV, test_csv_path=TEST_CSV_PATH):
    output_df = pd.read_csv(output_csv_path)
    test_df = pd.read_csv(test_csv_path)
    expected_columns = [ID_COLUMN] + TARGET_COLUMNS
    assert list(output_df.columns) == expected_columns, f"Unexpected columns: {list(output_df.columns)}"
    assert len(output_df) == len(test_df), f"Row count mismatch: output={len(output_df)}, test={len(test_df)}"
    assert not output_df[ID_COLUMN].duplicated().any(), "Duplicate ids in output."
    assert output_df[ID_COLUMN].tolist() == test_df[ID_COLUMN].tolist(), "Output id order differs from test CSV."
    print("Output CSV format is valid.")
    return output_df


validated_submission = validate_output_csv(SUBMISSION_CSV, TEST_CSV_PATH)
